<a href="https://colab.research.google.com/github/Yukselendincer/Microsoft-Data-Science/blob/main/YapayZeka_Destekli_COSO_i%C3%A7_denetim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-google-genai langchain-community faiss-cpu sentence-transformers langchain-huggingface duckduckgo-search ddgs langgraph

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Colab Secrets'tan anahtarı alıyoruz
os.environ["GOOGLE_API_KEY"] = userdata.get('coso1')

# DÜZELTME: API'nin yönlendirdiği en güncel model olan 'gemini-3.8-flash' kullanıyoruz.
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
print("Beyin (Gemini LLM) 'gemini-3.8-flash' modeli ile aktif edildi.")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import Tool

# COSO Çerçevesi metnini oluşturma
coso_metni = """COSO İç Kontrol Çerçevesi 5 temel bileşenden oluşur:
1. Kontrol Ortamı: Kurumun etik değerleri ve organizasyon yapısı.
2. Risk Değerlendirmesi: Hedeflere ulaşmayı engelleyecek iç ve dış risklerin analizi.
3. Kontrol Faaliyetleri: Riskleri azaltmaya yönelik politikalar ve prosedürler.
4. Bilgi ve İletişim: Gerekli bilgilerin doğru zamanda, doğru kişilere iletilmesi.
5. İzleme Faaliyetleri: İç kontrol sisteminin performansının sürekli değerlendirilmesi.
İç denetim fonksiyonu, organizasyonun COSO standartlarına uyumunu test eder ve raporlar."""

with open("coso_rehberi.txt", "w", encoding="utf-8") as f:
    f.write(coso_metni)

# Belgeyi yükleme ve parçalama
loader = TextLoader("coso_rehberi.txt")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
splits = text_splitter.split_documents(docs)

# Yerel Hugging Face modeli ile vektör veritabanını oluşturma
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever()

# RAG aracını tanımlama
def rag_sorgusu(query: str) -> str:
    sonuclar = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in sonuclar])

rag_araci = Tool(
    name="coso_ic_denetim_rehberi",
    description="Kurumun iç denetim standartları, COSO bileşenleri ve kontrol prosedürleri hakkında bilgi ararken bu aracı kullan.",
    func=rag_sorgusu
)
print("COSO RAG (Bilgi Kaynağı) aracı oluşturuldu.")

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# Dış dünya ile bağlantı kuracak web arama aracı
web_arama_araci = DuckDuckGoSearchRun(
    name="mevzuat_ve_web_arama",
    description="Güncel dış denetim haberleri, yeni mevzuatlar veya yasal güncellemeler hakkında bilgi ararken bu aracı kullan."
)

# Ajanın kullanabileceği araçların listesi
tools = [rag_araci, web_arama_araci]
print("Araçlar (Sinir Sistemi) ajana bağlandı.")

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

sistem_mesaji = "Sen uzman bir İç Denetim ve Risk Yönetimi asistanısın. COSO İç Kontrol Çerçevesi standartlarına hakimsin. İç kurallar veya COSO bileşenleri sorulduğunda 'coso_ic_denetim_rehberi' aracını kullan. Dış dünyadaki güncel denetim haberleri veya yasal mevzuat değişiklikleri sorulduğunda 'mevzuat_ve_web_arama' aracını kullan."

# Ajanı başlatma (Deprecation uyarılarını göz ardı ediyoruz)
agent_executor = create_react_agent(llm, tools=tools, prompt=sistem_mesaji)

print("\n--- Test 1: Kurum İçi / COSO Bilgisi ---")
mesaj_1 = {"messages": [HumanMessage(content="Yeni bir süreci denetliyorum. COSO'nun risk değerlendirmesi bileşeni neleri kapsar?")]}
cevap_1 = agent_executor.invoke(mesaj_1)
print("\nAJANIN CEVABI 1:\n", cevap_1["messages"][-1].content)

print("\n--- Test 2: Web / Mevzuat Araması ---")
mesaj_2 = {"messages": [HumanMessage(content="2026 yılında iç denetimde yapay zeka kullanımı veya siber güvenlik riskleri ile ilgili güncel gelişmeler nelerdir?")]}
cevap_2 = agent_executor.invoke(mesaj_2)
print("\nAJANIN CEVABI 2:\n", cevap_2["messages"][-1].content)